# 🛡️ RAGGuard — Main Pipeline (`00_MAIN.ipynb`)
**Trustworthy AI mini-project — Customer-service RAG chatbot: Evaluate → Attack → Defend.**

This notebook runs the whole graded pipeline end-to-end and produces every table/plot
for the report. Target: Colab with an L4 GPU. Set `FAST=True` for a ~8-min validation
run; `FAST=False` for the numbers that go in the report.


In [ ]:
# === Setup - idempotent, safe to re-run. Works on Colab and locally. ===
# On Colab this clones the repo into the VM and installs deps; locally it's a no-op
# beyond adding the repo to sys.path (assumes you've already created the venv).
import os, sys, pathlib, subprocess

REPO_URL = "https://github.com/maxtonhuang/Week6_Mini_Project_Customer_Service_RAG_Support_Chatbot.git"
REPO_DIR = pathlib.Path("/content") / "Week6_Mini_Project_Customer_Service_RAG_Support_Chatbot"

# 1) Ensure the repo files are present and become the working directory.
if pathlib.Path("ragguard").is_dir():
    pass                                    # already inside the repo (local, or an earlier run)
elif (REPO_DIR / "ragguard").is_dir():
    os.chdir(REPO_DIR)                       # cloned earlier this session -> reuse it
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)                       # first run on this VM -> clone it

REPO = pathlib.Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("working dir:", REPO)

# 2) On Colab only, install the Python dependencies (no-op elsewhere).
try:
    import google.colab  # noqa: F401
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # Optional: persist the ~16 GB model cache across sessions via Google Drive:
    # from google.colab import drive; drive.mount("/content/drive")
    # os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
except ImportError:
    pass


In [ ]:
# --- Auto-configure for the available GPU: picks bf16 / 4-bit / smaller model by VRAM
#     and pip-installs bitsandbytes only if a 4-bit path is chosen ---
from ragguard import autotune
autotune.apply()                 # sets GEN_MODEL / LOAD_IN_4BIT / BATCH_SIZE automatically

# --- Runtime dials (override autotune below if you want) ---
FAST = True                      # True = quick validation; False = full report run
import ragguard.config as C
C.N_PER_ATTACK   = 8  if FAST else 50
C.ADAPTIVE_ROUNDS= 3  if FAST else 6
C.SCREEN_N       = 6  if FAST else 15
# C.BATCH_SIZE = 2       # force a smaller batch if you still hit CUDA OOM
SEED = C.SEED
print(f"FAST={FAST}  model={C.GEN_MODEL}  4bit={C.LOAD_IN_4BIT}  batch={C.BATCH_SIZE}  device={C.device()}")


## §1 — System build & threat model (Criterion 1)


In [ ]:
from ragguard import corpus, prompts, rag
from ragguard import canary as canary_mod

docs, benign = corpus.build_knowledge_base(seed=SEED)
canaries = [d for d in docs if d.is_canary()]
canary_mod.save_registry(canaries)
print(f"Knowledge base: {len(docs)} docs  ({len(canaries)} planted canaries)")
print(f"Benign eval set: {len(benign)} held-out Q&A")
print("Example canary token:", canaries[0].canary)


In [ ]:
# Build the victim RAG pipeline. Wrap the model in CachedLLM so the 64-stack search in
# §3 reuses generations instead of recomputing them (turns a ~hour into minutes).
from ragguard.cache import CachedLLM
llm = CachedLLM(rag.QwenLLM(max_new_tokens=C.EVAL_MAX_NEW_TOKENS))
retriever = rag.EmbeddingRetriever(docs)
pipe = rag.RagPipeline(retriever, llm)

# Sanity check: a benign question should get a helpful answer
resp = pipe.answer("How long do I have to return an item?")
print("A:", resp.answer)
print("retrieved:", [d.doc_id for d in resp.retrieved])


In [ ]:
# NIST AI RMF baseline scorecard (undefended system)
from ragguard import governance
from IPython.display import Markdown, display
baseline = governance.baseline_scorecard()
display(Markdown(governance.render_markdown(baseline)))


## §2 — Attacks & evidence (Criterion 2)


In [ ]:
from ragguard.judge import RuleJudge
from ragguard.attacks import build_all_attacks
from ragguard.attacks.base import load_hf_prompts
from ragguard import orchestrator, metrics, report

# Optionally load real injection/jailbreak prompts from HF (falls back to built-ins offline)
inj = load_hf_prompts(C.INJECTION_DATASET, "train", "text", "label", 1)
jbk = load_hf_prompts(C.JAILBREAK_DATASET, "train", "prompt", "type", "jailbreak")

judge = RuleJudge(canary_tokens=canary_mod.canary_tokens(canaries),
                  system_prompt_secrets=prompts.SYSTEM_PROMPT_SECRETS)
attacks = build_all_attacks(canary_docs=canaries, injection_prompts=inj, jailbreak_prompts=jbk)

undef = orchestrator.run_suite(pipe, attacks, judge, defenses=None)
print(f"Undefended overall ASR = {metrics.asr(undef):.0%}")
for a, v in sorted(metrics.group_asr(undef, 'attack_id').items()):
    print(f"  {a}: {v:.0%}")


In [ ]:
# ASR bar chart + results table
report.asr_bar(undef, C.artifact_dir() / "asr_undefended.png")
print(report.results_table(undef)[:1500])


In [ ]:
# A7 — adaptive attacker agent (mutate-retry loop). Uses Qwen as the attacker on Colab.
from ragguard.attacks import AdaptiveAttacker, HeuristicAttackerLLM, adaptive_asr_curve
attacker = HeuristicAttackerLLM()     # swap for rag.QwenLLM() for an LLM-driven attacker
seeds = [c for atk in attacks for c in atk.generate(3)]
adv = AdaptiveAttacker(attacker, rounds=C.ADAPTIVE_ROUNDS)
adv_recs = adv.run(seeds, pipe, judge)
curve = adaptive_asr_curve(adv_recs)
print("Adaptive ASR by round:", [f"{x:.0%}" for x in curve])
report.adaptive_curve_plot(curve, C.artifact_dir() / "adaptive_curve.png")


## §3 — Trustworthy AI design (Criterion 3)


In [ ]:
from ragguard.defenses import build_all_defenses
defenses = build_all_defenses(system_prompt_secrets=prompts.SYSTEM_PROMPT_SECRETS)

# Attack x defense matrix -> heatmap
matrix = orchestrator.attack_defense_matrix(pipe, attacks, judge, defenses)
report.attack_defense_heatmap(matrix, C.artifact_dir() / "heatmap.png")
print(f"Full-stack ASR = {metrics.asr([r for r in matrix if '+' in r.defense_stack]):.0%}")


In [ ]:
# Two-stage search over all 64 defense stacks -> best tradeoff
search = orchestrator.two_stage_search(pipe, attacks, judge, defenses, benign)
best = search['best']
print(f"BEST STACK: [{best['stack']}]  robustness={best['robustness']:.0%} "
      f"utility={best['utility']:.2f} FRR={best['frr']:.0%}")
report.pareto_plot(search['pareto'], C.artifact_dir() / "pareto.png", knee=search['knee'])


In [ ]:
# NIST AI RMF re-score (defended) + before/after comparison
undef_asr = metrics.asr(undef)
defended = governance.defended_scorecard({
    "baseline_asr": undef_asr, "defended_asr": best['asr'],
    "best_stack": best['stack'], "frr": best['frr']})
display(Markdown(governance.compare(baseline, defended)))


## §4 — Persist artefacts (for `01_DEMO.ipynb` and the report)


In [ ]:
import json
results = {
    "baseline_asr": metrics.asr(undef),
    "defended_asr": best['asr'],
    "best_stack": best['stack'],
    "frr": best['frr'],
    "utility": best['utility'],
    "asr_by_attack": metrics.group_asr(undef, 'attack_id'),
    "adaptive_curve": curve,
    "pareto": [list(p) for p in search['pareto']],
}
(C.artifact_dir() / "results.json").write_text(json.dumps(results, indent=2))
report.records_to_csv(undef, C.artifact_dir() / "records_undefended.csv")
report.records_to_csv(matrix, C.artifact_dir() / "records_matrix.csv")
print("Saved artefacts to", C.artifact_dir())
